In [1]:
from pathlib import Path
import time
import json
import numpy as np

from pynq import Overlay, allocate
from pynq.pmbus import get_rails, DataRecorder

BASE = Path("/home/xilinx/jupyter_notebooks/resnet8_hls_ip11")

BIT = BASE / "resnet8_hls_ip11.bit"
DATA = BASE / "cifar10_test_uint8.npz"

overlay = Overlay(str(BIT), download=True)
dma = overlay.axi_dma_0

data = np.load(DATA)
x_test = data["x"]
y_test = data["y"].reshape(-1)

print("Dataset:", x_test.shape, x_test.dtype)
print("Classe imagem 0:", int(y_test[0]))
print("DMA carregado.")

Dataset: (10000, 32, 32, 3) uint8
Classe imagem 0: 3
DMA carregado.


In [2]:
FIXED_W = 22
FIXED_I = 12
FIXED_F = 10

SCALE = 1 << FIXED_F
MASK22 = (1 << FIXED_W) - 1
SIGN22 = 1 << (FIXED_W - 1)

RAW_MIN = -(1 << (FIXED_W - 1))
RAW_MAX = (1 << (FIXED_W - 1)) - 1

input_buffer = allocate(
    shape=(1024, 4),
    dtype=np.uint32
)

output_buffer = allocate(
    shape=(16,),
    dtype=np.uint32
)


def pack_image_u8(image_u8, destination):
    x = image_u8.astype(np.float32) / np.float32(255.0)

    q = np.rint(
        x * np.float32(SCALE)
    ).astype(np.int64)

    q = np.clip(
        q,
        RAW_MIN,
        RAW_MAX
    )

    q = (
        q & MASK22
    ).astype(np.uint32)

    q = q.reshape(1024, 3)

    destination[:, 0] = q[:, 0]
    destination[:, 1] = q[:, 1]
    destination[:, 2] = q[:, 2]
    destination[:, 3] = 0


def dma_inference_only():
    dma.recvchannel.transfer(
        output_buffer,
        nbytes=64
    )

    dma.sendchannel.transfer(
        input_buffer,
        nbytes=16384
    )

    dma.sendchannel.wait()
    dma.recvchannel.wait()


def decode_logits():
    raw = (
        np.asarray(
            output_buffer[:10],
            dtype=np.uint32
        ) & MASK22
    ).astype(np.int64)

    negative = (
        raw & SIGN22
    ) != 0

    raw[negative] -= (
        1 << FIXED_W
    )

    return (
        raw.astype(np.float32)
        / np.float32(SCALE)
    )

In [3]:
pack_image_u8(
    x_test[0],
    input_buffer
)

dma_inference_only()

logits = decode_logits()
pred = int(np.argmax(logits))

print("Esperada:", int(y_test[0]))
print("FPGA    :", pred)
print("Logits  :", logits)
print("Padding :", output_buffer[10:16])
print(
    "DMA errors:",
    dma.sendchannel.error,
    dma.recvchannel.error
)

Esperada: 3
FPGA    : 3
Logits  : [ -7.7910156  -8.96582   -13.777344    3.600586  -10.802734    1.8779297
  -1.3779297 -13.140625   -5.9716797 -12.662109 ]
Padding : [0 0 0 0 0 0]
DMA errors: False False


In [4]:
WARMUP = 100
ACTIVE_SECONDS = 20.0

pack_image_u8(
    x_test[0],
    input_buffer
)

print("Warm-up...")

for _ in range(WARMUP):
    dma_inference_only()

print("Medindo throughput saturado...")

count = 0

t0 = time.perf_counter()

while (
    time.perf_counter() - t0
    < ACTIVE_SECONDS
):
    dma_inference_only()
    count += 1

elapsed = (
    time.perf_counter() - t0
)

fps_saturated = (
    count / elapsed
)

print()
print(
    "Inferências:",
    count
)

print(
    "Tempo:",
    f"{elapsed:.6f} s"
)

print(
    "Throughput saturado:",
    f"{fps_saturated:.3f} FPS"
)

print(
    "Tempo equivalente:",
    f"{1000/fps_saturated:.6f} ms/inf"
)

print(
    "DMA errors:",
    dma.sendchannel.error,
    dma.recvchannel.error
)

Warm-up...
Medindo throughput saturado...

Inferências: 24021
Tempo: 20.000652 s
Throughput saturado: 1201.011 FPS
Tempo equivalente: 0.832632 ms/inf
DMA errors: False False


In [5]:
N_LAT = 10000

lat_ms = np.empty(
    N_LAT,
    dtype=np.float64
)

pack_image_u8(
    x_test[0],
    input_buffer
)

for _ in range(100):
    dma_inference_only()

for i in range(N_LAT):

    t0 = time.perf_counter_ns()

    dma_inference_only()

    t1 = time.perf_counter_ns()

    lat_ms[i] = (
        t1 - t0
    ) / 1_000_000.0


print(
    "Latência média :",
    f"{np.mean(lat_ms):.6f} ms"
)

print(
    "Mediana        :",
    f"{np.median(lat_ms):.6f} ms"
)

print(
    "Desvio         :",
    f"{np.std(lat_ms, ddof=1):.6f} ms"
)

print(
    "p95            :",
    f"{np.percentile(lat_ms,95):.6f} ms"
)

print(
    "p99            :",
    f"{np.percentile(lat_ms,99):.6f} ms"
)

print(
    "Min            :",
    f"{np.min(lat_ms):.6f} ms"
)

print(
    "Max            :",
    f"{np.max(lat_ms):.6f} ms"
)

print()
print(
    "1000 / latência média:",
    f"{1000/np.mean(lat_ms):.3f} FPS"
)

Latência média : 0.830158 ms
Mediana        : 0.831830 ms
Desvio         : 0.007152 ms
p95            : 0.842351 ms
p99            : 0.850160 ms
Min            : 0.818170 ms
Max            : 0.900760 ms

1000 / latência média: 1204.590 FPS


In [6]:
rails = get_rails()

power_sensor = rails["12V"].power

print(
    "Sensor:",
    power_sensor.name
)

print(
    "Potência atual:",
    power_sensor.value,
    "W"
)

Sensor: 12V_power
Potência atual: 10.85 W


In [7]:
rails = get_rails()

power_sensor = rails["12V"].power

print(
    "Sensor:",
    power_sensor.name
)

print(
    "Potência atual:",
    power_sensor.value,
    "W"
)

Sensor: 12V_power
Potência atual: 11.262 W


In [8]:
idle_recorder = DataRecorder(
    power_sensor
)

with idle_recorder.record(0.1):
    time.sleep(5.0)

idle_frame = idle_recorder.frame

idle_power = float(
    idle_frame[
        power_sensor.name
    ].mean()
)

print(
    "Idle power:",
    f"{idle_power:.4f} W"
)

print(
    "Amostras:",
    len(idle_frame)
)

Idle power: 10.5908 W
Amostras: 46


In [9]:
ACTIVE_SECONDS = 20.0

active_recorder = DataRecorder(
    power_sensor
)

count = 0

with active_recorder.record(0.1):

    t0 = time.perf_counter()

    while (
        time.perf_counter() - t0
        < ACTIVE_SECONDS
    ):
        dma_inference_only()
        count += 1

    elapsed = (
        time.perf_counter() - t0
    )

active_frame = active_recorder.frame

active_power = float(
    active_frame[
        power_sensor.name
    ].mean()
)

fps_sat = (
    count / elapsed
)

dynamic_power = max(
    active_power - idle_power,
    0
)

energy_total_mj = (
    active_power
    / fps_sat
    * 1000
)

energy_dynamic_mj = (
    dynamic_power
    / fps_sat
    * 1000
)

print(
    "FPS saturado       :",
    f"{fps_sat:.3f}"
)

print(
    "Idle power         :",
    f"{idle_power:.4f} W"
)

print(
    "Active power       :",
    f"{active_power:.4f} W"
)

print(
    "Dynamic power      :",
    f"{dynamic_power:.4f} W"
)

print(
    "Energia total/inf  :",
    f"{energy_total_mj:.4f} mJ"
)

print(
    "Energia dinâmica   :",
    f"{energy_dynamic_mj:.4f} mJ"
)

print(
    "Power samples      :",
    len(active_frame)
)

FPS saturado       : 1113.723
Idle power         : 10.5908 W
Active power       : 12.6996 W
Dynamic power      : 2.1088 W
Energia total/inf  : 11.4028 mJ
Energia dinâmica   : 1.8934 mJ
Power samples      : 164


In [10]:
from datetime import datetime, timezone

RESULT = {
    "experiment": (
        "saturated_hardware_path"
    ),

    "definition": (
        "same input already packed; "
        "repeated DMA MM2S -> "
        "hls4ml ResNet8 -> "
        "DMA S2MM"
    ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "batch": 1,

    "warmup": 100,

    "active_seconds":
        ACTIVE_SECONDS,

    "throughput_fps":
        fps_sat,

    "latency_mean_ms":
        float(np.mean(lat_ms)),

    "latency_median_ms":
        float(np.median(lat_ms)),

    "latency_p95_ms":
        float(
            np.percentile(
                lat_ms,
                95
            )
        ),

    "latency_p99_ms":
        float(
            np.percentile(
                lat_ms,
                99
            )
        ),

    "idle_power_w":
        idle_power,

    "active_power_w":
        active_power,

    "dynamic_power_w":
        dynamic_power,

    "energy_total_mj":
        energy_total_mj,

    "energy_dynamic_mj":
        energy_dynamic_mj,

    "power_sensor":
        power_sensor.name,

    "clock_mhz": 100.0,

    "precision":
        "ap_fixed<22,12,AP_RND_CONV,AP_SAT>"
}

OUT = (
    BASE
    / "resultados_robustos"
    / "hardware_path_saturated.json"
)

OUT.write_text(
    json.dumps(
        RESULT,
        indent=2
    )
)

np.save(
    BASE
    / "resultados_robustos"
    / "hardware_path_saturated_latencies.npy",
    lat_ms
)

print(
    "Salvo em:",
    OUT
)

Salvo em: /home/xilinx/jupyter_notebooks/resnet8_hls_ip11/resultados_robustos/hardware_path_saturated.json


In [11]:
import csv
import math
import json
from datetime import datetime, timezone
from pathlib import Path

SEED = 20260825

N_IMAGES = 5000
N_CYCLES = 20
WARMUP = 100

SAT_WINDOWS = 8
SAT_SECONDS = 10.0

POWER_WINDOWS = 8
POWER_SECONDS = 10.0
POWER_INTERVAL = 0.1

IDLE_SECONDS = 10.0

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

OUT = (
    BASE
    / f"resultados_intermediario_{RUN_ID}"
)

OUT.mkdir(parents=True)

print("Resultados serão salvos em:")
print(OUT)

Resultados serão salvos em:
/home/xilinx/jupyter_notebooks/resnet8_hls_ip11/resultados_intermediario_20250504_103336


In [12]:
rng = np.random.default_rng(SEED)

subset_parts = []

for c in range(10):

    idx = np.flatnonzero(
        y_test == c
    ).copy()

    rng.shuffle(idx)

    subset_parts.append(
        idx[:500]
    )

subset_idx = np.concatenate(
    subset_parts
)

rng.shuffle(subset_idx)

assert len(subset_idx) == 5000

distribution = np.bincount(
    y_test[subset_idx],
    minlength=10
)

print("Distribuição:")
print(distribution)

np.save(
    OUT / "subset_indices.npy",
    subset_idx
)

Distribuição:
[500 500 500 500 500 500 500 500 500 500]


In [13]:
packed_subset = np.zeros(
    (N_IMAGES, 1024, 4),
    dtype=np.uint32
)

print("Pré-empacotando...")

for i, idx in enumerate(subset_idx):

    img = x_test[idx]

    x = (
        img.astype(np.float32)
        / np.float32(255.0)
    )

    q = np.rint(
        x * np.float32(SCALE)
    ).astype(np.int64)

    q = np.clip(
        q,
        RAW_MIN,
        RAW_MAX
    )

    q = (
        q & MASK22
    ).astype(np.uint32)

    q = q.reshape(1024, 3)

    packed_subset[i, :, 0] = q[:, 0]
    packed_subset[i, :, 1] = q[:, 1]
    packed_subset[i, :, 2] = q[:, 2]
    packed_subset[i, :, 3] = 0

print(
    "Tamanho:",
    f"{packed_subset.nbytes / 1024**2:.2f} MiB"
)

Pré-empacotando...
Tamanho: 78.12 MiB


In [14]:
REF_FILE = (
    BASE
    / "resultados_robustos"
    / "predictions_accuracy_10000.npy"
)

if REF_FILE.exists():

    ref_full = np.load(
        REF_FILE
    )

    assert ref_full.shape == (10000,)

    ref_subset = ref_full[
        subset_idx
    ]

    print(
        "Referência carregada do "
        "benchmark de 10.000 imagens."
    )

else:

    print(
        "Referência antiga não encontrada; "
        "gerando agora..."
    )

    ref_subset = np.empty(
        N_IMAGES,
        dtype=np.uint8
    )

    for i in range(N_IMAGES):

        input_buffer[:] = (
            packed_subset[i]
        )

        dma_inference_only()

        ref_subset[i] = int(
            np.argmax(
                decode_logits()
            )
        )

np.save(
    OUT / "reference_predictions.npy",
    ref_subset
)

correct_subset = int(
    np.sum(
        ref_subset
        == y_test[subset_idx]
    )
)

print(
    "Acurácia do subset:",
    f"{correct_subset/N_IMAGES*100:.3f}%"
)

Referência carregada do benchmark de 10.000 imagens.
Acurácia do subset: 74.680%


In [15]:
Z95 = 1.959963984540054


def bootstrap_ci_mean(
    values,
    seed=SEED,
    resamples=10000
):
    values = np.asarray(
        values,
        dtype=np.float64
    )

    rng = np.random.default_rng(
        seed
    )

    n = len(values)

    means = np.empty(
        resamples,
        dtype=np.float64
    )

    for i in range(resamples):

        sample = rng.choice(
            values,
            size=n,
            replace=True
        )

        means[i] = np.mean(sample)

    return (
        float(
            np.percentile(
                means,
                2.5
            )
        ),
        float(
            np.percentile(
                means,
                97.5
            )
        )
    )


def cycle_statistics(
    values,
    seed=SEED
):
    values = np.asarray(
        values,
        dtype=np.float64
    )

    mean = float(
        np.mean(values)
    )

    sd = float(
        np.std(
            values,
            ddof=1
        )
    )

    cv = (
        sd / mean * 100
    )

    half = (
        Z95
        * sd
        / np.sqrt(len(values))
    )

    boot_low, boot_high = (
        bootstrap_ci_mean(
            values,
            seed=seed
        )
    )

    k = min(
        5,
        len(values) // 2
    )

    first = float(
        np.mean(
            values[:k]
        )
    )

    last = float(
        np.mean(
            values[-k:]
        )
    )

    drift = (
        (last - first)
        / first
        * 100
    )

    return {
        "mean": mean,
        "sd": sd,
        "cv_percent": cv,

        "ci95_low":
            mean - half,

        "ci95_high":
            mean + half,

        "bootstrap_ci95_low":
            boot_low,

        "bootstrap_ci95_high":
            boot_high,

        "first5_mean":
            first,

        "last5_mean":
            last,

        "drift_percent":
            drift
    }


def latency_statistics(
    lat_matrix
):

    flat = np.asarray(
        lat_matrix,
        dtype=np.float64
    ).reshape(-1)

    cycle_means = (
        np.mean(
            lat_matrix,
            axis=1
        )
    )

    return {
        "samples":
            int(flat.size),

        "mean_ms":
            float(np.mean(flat)),

        "median_ms":
            float(np.median(flat)),

        "std_ms":
            float(
                np.std(
                    flat,
                    ddof=1
                )
            ),

        "cv_percent":
            float(
                np.std(
                    flat,
                    ddof=1
                )
                / np.mean(flat)
                * 100
            ),

        "p95_ms":
            float(
                np.percentile(
                    flat,
                    95
                )
            ),

        "p99_ms":
            float(
                np.percentile(
                    flat,
                    99
                )
            ),

        "min_ms":
            float(np.min(flat)),

        "max_ms":
            float(np.max(flat)),

        "cycle_means":
            cycle_statistics(
                cycle_means
            )
    }

In [16]:
lat_inf = np.empty(
    (N_CYCLES, N_IMAGES),
    dtype=np.float32
)

fps_inf_effective = np.empty(
    N_CYCLES,
    dtype=np.float64
)

fps_inf_path = np.empty(
    N_CYCLES,
    dtype=np.float64
)

print("Warm-up inference-only...")

for i in range(WARMUP):

    input_buffer[:] = (
        packed_subset[
            i % N_IMAGES
        ]
    )

    dma_inference_only()


for cycle in range(N_CYCLES):

    rng = np.random.default_rng(
        SEED + cycle
    )

    order = rng.permutation(
        N_IMAGES
    )

    mismatches = 0

    wall0 = time.perf_counter()

    for j, pos in enumerate(order):

        # Fora da latência individual.
        input_buffer[:] = (
            packed_subset[pos]
        )

        t0 = time.perf_counter_ns()

        dma_inference_only()

        t1 = time.perf_counter_ns()

        lat_inf[cycle, j] = (
            t1 - t0
        ) / 1_000_000

        # Fora da latência individual,
        # mas dentro do wall-time efetivo.
        pred = int(
            np.argmax(
                decode_logits()
            )
        )

        if pred != int(
            ref_subset[pos]
        ):
            mismatches += 1

    wall = (
        time.perf_counter()
        - wall0
    )

    if mismatches != 0:
        raise RuntimeError(
            f"Ciclo {cycle}: "
            f"{mismatches} divergências"
        )

    if (
        dma.sendchannel.error
        or dma.recvchannel.error
    ):
        raise RuntimeError(
            "Erro DMA"
        )

    mean_lat = float(
        np.mean(
            lat_inf[cycle]
        )
    )

    fps_inf_path[cycle] = (
        1000.0 / mean_lat
    )

    fps_inf_effective[cycle] = (
        N_IMAGES / wall
    )

    print(
        f"{cycle+1:02d}/{N_CYCLES} | "
        f"lat={mean_lat:.4f} ms | "
        f"path={fps_inf_path[cycle]:.2f} FPS | "
        f"effective={fps_inf_effective[cycle]:.2f} FPS"
    )

Warm-up inference-only...
01/20 | lat=0.8445 ms | path=1184.15 FPS | effective=901.42 FPS
02/20 | lat=0.8463 ms | path=1181.67 FPS | effective=897.84 FPS
03/20 | lat=0.8463 ms | path=1181.56 FPS | effective=900.46 FPS
04/20 | lat=0.8452 ms | path=1183.13 FPS | effective=900.64 FPS
05/20 | lat=0.8473 ms | path=1180.29 FPS | effective=899.83 FPS
06/20 | lat=0.8442 ms | path=1184.56 FPS | effective=901.48 FPS
07/20 | lat=0.8457 ms | path=1182.44 FPS | effective=900.83 FPS
08/20 | lat=0.8458 ms | path=1182.34 FPS | effective=899.70 FPS
09/20 | lat=0.8443 ms | path=1184.40 FPS | effective=902.94 FPS
10/20 | lat=0.8451 ms | path=1183.34 FPS | effective=900.82 FPS
11/20 | lat=0.8446 ms | path=1183.98 FPS | effective=902.17 FPS
12/20 | lat=0.8439 ms | path=1184.92 FPS | effective=902.74 FPS
13/20 | lat=0.8452 ms | path=1183.10 FPS | effective=901.32 FPS
14/20 | lat=0.8454 ms | path=1182.91 FPS | effective=900.57 FPS
15/20 | lat=0.8452 ms | path=1183.22 FPS | effective=899.86 FPS
16/20 | lat=0.

In [17]:
lat_e2e = np.empty(
    (N_CYCLES, N_IMAGES),
    dtype=np.float32
)

fps_e2e = np.empty(
    N_CYCLES,
    dtype=np.float64
)

print("Warm-up end-to-end...")

for i in range(WARMUP):

    pos = i % N_IMAGES

    pack_image_u8(
        x_test[
            subset_idx[pos]
        ],
        input_buffer
    )

    dma_inference_only()

    _ = int(
        np.argmax(
            decode_logits()
        )
    )


for cycle in range(N_CYCLES):

    rng = np.random.default_rng(
        SEED + 1000 + cycle
    )

    order = rng.permutation(
        N_IMAGES
    )

    mismatches = 0

    wall0 = time.perf_counter()

    for j, pos in enumerate(order):

        t0 = time.perf_counter_ns()

        pack_image_u8(
            x_test[
                subset_idx[pos]
            ],
            input_buffer
        )

        dma_inference_only()

        pred = int(
            np.argmax(
                decode_logits()
            )
        )

        t1 = time.perf_counter_ns()

        lat_e2e[cycle, j] = (
            t1 - t0
        ) / 1_000_000

        if pred != int(
            ref_subset[pos]
        ):
            mismatches += 1

    wall = (
        time.perf_counter()
        - wall0
    )

    if mismatches:
        raise RuntimeError(
            f"Ciclo {cycle}: "
            f"{mismatches} divergências"
        )

    fps_e2e[cycle] = (
        N_IMAGES / wall
    )

    print(
        f"{cycle+1:02d}/{N_CYCLES} | "
        f"lat="
        f"{np.mean(lat_e2e[cycle]):.4f} ms | "
        f"FPS={fps_e2e[cycle]:.2f}"
    )

Warm-up end-to-end...
01/20 | lat=1.6012 ms | FPS=619.30
02/20 | lat=1.6033 ms | FPS=618.58
03/20 | lat=1.5934 ms | FPS=622.40
04/20 | lat=1.5977 ms | FPS=620.74
05/20 | lat=1.5978 ms | FPS=620.60
06/20 | lat=1.5945 ms | FPS=621.92
07/20 | lat=1.5988 ms | FPS=620.32
08/20 | lat=1.5998 ms | FPS=619.84
09/20 | lat=1.5939 ms | FPS=622.23
10/20 | lat=1.5971 ms | FPS=620.90
11/20 | lat=1.5992 ms | FPS=620.16
12/20 | lat=1.5963 ms | FPS=621.26
13/20 | lat=1.5988 ms | FPS=620.22
14/20 | lat=1.5985 ms | FPS=620.45
15/20 | lat=1.5920 ms | FPS=622.87
16/20 | lat=1.6021 ms | FPS=619.03
17/20 | lat=1.5961 ms | FPS=621.29
18/20 | lat=1.5982 ms | FPS=620.63
19/20 | lat=1.5987 ms | FPS=620.25
20/20 | lat=1.5967 ms | FPS=621.07


In [18]:
fps_saturated_runs = np.empty(
    SAT_WINDOWS,
    dtype=np.float64
)

input_buffer[:] = (
    packed_subset[0]
)

for _ in range(WARMUP):
    dma_inference_only()


for run in range(SAT_WINDOWS):

    count = 0

    t0 = time.perf_counter()

    while (
        time.perf_counter() - t0
        < SAT_SECONDS
    ):

        dma_inference_only()

        count += 1

    elapsed = (
        time.perf_counter()
        - t0
    )

    fps_saturated_runs[run] = (
        count / elapsed
    )

    print(
        f"{run+1:02d}/{SAT_WINDOWS} | "
        f"{fps_saturated_runs[run]:.3f} FPS"
    )

01/8 | 1206.252 FPS
02/8 | 1199.613 FPS
03/8 | 1197.671 FPS
04/8 | 1199.057 FPS
05/8 | 1205.879 FPS
06/8 | 1199.672 FPS
07/8 | 1199.686 FPS
08/8 | 1199.691 FPS


In [19]:
sat_stats = cycle_statistics(
    fps_saturated_runs,
    seed=SEED + 2000
)

sat_stats

{'mean': 1200.9402231905733,
 'sd': 3.2371184695009956,
 'cv_percent': 0.2695486758617217,
 'ci95_low': 1198.697055757196,
 'ci95_high': 1203.1833906239506,
 'bootstrap_ci95_low': 1199.0859853898462,
 'bootstrap_ci95_high': 1203.237847526764,
 'first5_mean': 1200.648309703625,
 'last5_mean': 1201.2321366775218,
 'drift_percent': 0.048625977247314295}

In [20]:
inf_lat_stats = (
    latency_statistics(
        lat_inf
    )
)

e2e_lat_stats = (
    latency_statistics(
        lat_e2e
    )
)

inf_effective_stats = (
    cycle_statistics(
        fps_inf_effective,
        SEED + 3000
    )
)

inf_path_stats = (
    cycle_statistics(
        fps_inf_path,
        SEED + 4000
    )
)

e2e_fps_stats = (
    cycle_statistics(
        fps_e2e,
        SEED + 5000
    )
)

In [21]:
print("===== INFERENCE ONLY =====")

print(
    "N:",
    inf_lat_stats["samples"]
)

print(
    "Lat média:",
    inf_lat_stats["mean_ms"]
)

print(
    "Mediana:",
    inf_lat_stats["median_ms"]
)

print(
    "Desvio:",
    inf_lat_stats["std_ms"]
)

print(
    "CV:",
    inf_lat_stats["cv_percent"]
)

print(
    "p95:",
    inf_lat_stats["p95_ms"]
)

print(
    "p99:",
    inf_lat_stats["p99_ms"]
)

print(
    "Path FPS:",
    inf_path_stats
)

print(
    "Effective FPS:",
    inf_effective_stats
)


print("\n===== END TO END =====")

print(
    "N:",
    e2e_lat_stats["samples"]
)

print(
    "Lat média:",
    e2e_lat_stats["mean_ms"]
)

print(
    "Mediana:",
    e2e_lat_stats["median_ms"]
)

print(
    "Desvio:",
    e2e_lat_stats["std_ms"]
)

print(
    "CV:",
    e2e_lat_stats["cv_percent"]
)

print(
    "p95:",
    e2e_lat_stats["p95_ms"]
)

print(
    "p99:",
    e2e_lat_stats["p99_ms"]
)

print(
    "FPS:",
    e2e_fps_stats
)


print("\n===== SATURADO =====")

print(
    sat_stats
)

===== INFERENCE ONLY =====
N: 100000
Lat média: 0.8451455037820339
Mediana: 0.8456000089645386
Desvio: 0.014822770201774497
CV: 1.7538719824506506
p95: 0.8591600060462952
p99: 0.8759306108951566
Path FPS: {'mean': 1183.2296380785222, 'sd': 1.3335484532790132, 'cv_percent': 0.11270411172632538, 'ci95_low': 1182.6451954393972, 'ci95_high': 1183.8140807176471, 'bootstrap_ci95_low': 1182.6562937298145, 'bootstrap_ci95_high': 1183.786546288513, 'first5_mean': 1182.161351073777, 'last5_mean': 1183.7156325059411, 'drift_percent': 0.13147794340868182}
Effective FPS: {'mean': 901.0840920065248, 'sd': 1.3812547830767403, 'cv_percent': 0.1532881109909483, 'ci95_low': 900.4787415307018, 'ci95_high': 901.6894424823478, 'bootstrap_ci95_low': 900.4832278362951, 'bootstrap_ci95_high': 901.6732192832407, 'first5_mean': 900.0366994518296, 'last5_mean': 901.8136972762135, 'drift_percent': 0.19743615182205174}

===== END TO END =====
N: 100000
Lat média: 1.5976999370241165
Mediana: 1.593000054359436
Desvi

In [22]:
import csv
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone

# Dados brutos
np.save(OUT / "latencias_inference_only.npy", lat_inf)
np.save(OUT / "latencias_end_to_end.npy", lat_e2e)

np.save(
    OUT / "fps_inference_only_path.npy",
    fps_inf_path
)

np.save(
    OUT / "fps_inference_only_effective.npy",
    fps_inf_effective
)

np.save(
    OUT / "fps_end_to_end.npy",
    fps_e2e
)

np.save(
    OUT / "fps_saturated.npy",
    fps_saturated_runs
)

np.save(
    OUT / "subset_indices.npy",
    subset_idx
)

np.save(
    OUT / "reference_predictions.npy",
    ref_subset
)


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# CSV por ciclo
rows_inf = []

for i in range(N_CYCLES):

    x = lat_inf[i].astype(np.float64)

    rows_inf.append({
        "cycle": i + 1,
        "images": N_IMAGES,

        "latency_mean_ms":
            float(np.mean(x)),

        "latency_median_ms":
            float(np.median(x)),

        "latency_std_ms":
            float(np.std(x, ddof=1)),

        "latency_p95_ms":
            float(np.percentile(x, 95)),

        "latency_p99_ms":
            float(np.percentile(x, 99)),

        "latency_min_ms":
            float(np.min(x)),

        "latency_max_ms":
            float(np.max(x)),

        "path_fps":
            float(fps_inf_path[i]),

        "effective_fps":
            float(fps_inf_effective[i]),
    })


with open(
    OUT / "cycles_inference_only.csv",
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=rows_inf[0].keys()
    )

    writer.writeheader()
    writer.writerows(rows_inf)


rows_e2e = []

for i in range(N_CYCLES):

    x = lat_e2e[i].astype(np.float64)

    rows_e2e.append({
        "cycle": i + 1,
        "images": N_IMAGES,

        "latency_mean_ms":
            float(np.mean(x)),

        "latency_median_ms":
            float(np.median(x)),

        "latency_std_ms":
            float(np.std(x, ddof=1)),

        "latency_p95_ms":
            float(np.percentile(x, 95)),

        "latency_p99_ms":
            float(np.percentile(x, 99)),

        "latency_min_ms":
            float(np.min(x)),

        "latency_max_ms":
            float(np.max(x)),

        "effective_fps":
            float(fps_e2e[i]),
    })


with open(
    OUT / "cycles_end_to_end.csv",
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=rows_e2e[0].keys()
    )

    writer.writeheader()
    writer.writerows(rows_e2e)


# Resultado agregado
SUMMARY = {
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "platform":
        "AMD Xilinx ZCU104",

    "model":
        "ResNet8 CIFAR-10 hls4ml",

    "precision":
        "ap_fixed<22,12,AP_RND_CONV,AP_SAT>",

    "batch": 1,

    "subset_images":
        N_IMAGES,

    "cycles":
        N_CYCLES,

    "timed_inferences_per_scenario":
        N_IMAGES * N_CYCLES,

    "warmup":
        WARMUP,

    "seed":
        SEED,

    "inference_only": {
        "latency":
            inf_lat_stats,

        "path_fps":
            inf_path_stats,

        "effective_fps":
            inf_effective_stats,
    },

    "end_to_end": {
        "latency":
            e2e_lat_stats,

        "effective_fps":
            e2e_fps_stats,
    },

    "saturated": {
        "fps":
            sat_stats
    },

    "environment": {
        "python":
            sys.version,

        "numpy":
            np.__version__,

        "pynq":
            __import__("pynq").__version__,

        "kernel":
            platform.release(),

        "bit_sha256":
            sha256_file(BIT),

        "hwh_sha256":
            sha256_file(
                BIT.with_suffix(".hwh")
            ),

        "dataset_sha256":
            sha256_file(DATA),
    },

    "methodology": {
        "outlier_removal": False,

        "inference_only_latency":
            "PynqBuffer already contains packed input; "
            "timer covers DMA MM2S + accelerator + "
            "DMA S2MM + wait.",

        "effective_fps":
            "Full wall-time of cycle including input "
            "buffer copy, decode, argmax and Python loop.",

        "saturated_fps":
            "Same already-packed input repeatedly; "
            "only DMA + accelerator + DMA in loop.",

        "end_to_end":
            "Normalization + quantization/packing + "
            "DMA + accelerator + decode + argmax.",

        "ci":
            "95% CI over cycle/window-level observations.",

        "bootstrap":
            "10000 bootstrap resamples."
    }
}

with open(
    OUT / "SUMMARY.json",
    "w"
) as f:

    json.dump(
        SUMMARY,
        f,
        indent=2
    )


print("Tudo salvo em:")
print(OUT)

print("\nArquivos:")
for p in sorted(OUT.iterdir()):
    print(p.name)

Tudo salvo em:
/home/xilinx/jupyter_notebooks/resnet8_hls_ip11/resultados_intermediario_20250504_103336

Arquivos:
SUMMARY.json
cycles_end_to_end.csv
cycles_inference_only.csv
fps_end_to_end.npy
fps_inference_only_effective.npy
fps_inference_only_path.npy
fps_saturated.npy
latencias_end_to_end.npy
latencias_inference_only.npy
reference_predictions.npy
subset_indices.npy


In [23]:
TEST_INTERVALS = [
    0.1,
    0.2,
    0.5,
    1.0
]

TEST_SECONDS = 5.0
TEST_REPEATS = 3

telemetry_test = {}

input_buffer[:] = packed_subset[0]

for _ in range(100):
    dma_inference_only()


for interval in TEST_INTERVALS:

    values = []

    print(
        f"\nIntervalo = {interval}s"
    )

    for rep in range(TEST_REPEATS):

        recorder = DataRecorder(
            power_sensor
        )

        count = 0

        with recorder.record(interval):

            t0 = time.perf_counter()

            while (
                time.perf_counter()
                - t0
                < TEST_SECONDS
            ):

                dma_inference_only()

                count += 1

            elapsed = (
                time.perf_counter()
                - t0
            )

        fps = count / elapsed

        values.append(fps)

        print(
            f"{rep+1}/{TEST_REPEATS}: "
            f"{fps:.2f} FPS | "
            f"samples={len(recorder.frame)}"
        )

    telemetry_test[interval] = {
        "mean_fps":
            float(np.mean(values)),

        "sd_fps":
            float(
                np.std(
                    values,
                    ddof=1
                )
            ),

        "slowdown_percent":
            float(
                (
                    np.mean(values)
                    / sat_stats["mean"]
                    - 1
                ) * 100
            )
    }


print("\n===== RESUMO =====")

for interval, result in telemetry_test.items():

    print(
        f"{interval:>4}s | "
        f"{result['mean_fps']:.2f} FPS | "
        f"overhead="
        f"{result['slowdown_percent']:.2f}%"
    )


Intervalo = 0.1s
1/3: 1105.16 FPS | samples=42
2/3: 1105.47 FPS | samples=42
3/3: 1104.30 FPS | samples=42

Intervalo = 0.2s
1/3: 1144.73 FPS | samples=23
2/3: 1151.72 FPS | samples=23
3/3: 1152.84 FPS | samples=23

Intervalo = 0.5s
1/3: 1179.95 FPS | samples=10
2/3: 1179.64 FPS | samples=10
3/3: 1183.45 FPS | samples=10

Intervalo = 1.0s
1/3: 1195.30 FPS | samples=5
2/3: 1188.88 FPS | samples=5
3/3: 1188.83 FPS | samples=5

===== RESUMO =====
 0.1s | 1104.98 FPS | overhead=-7.99%
 0.2s | 1149.77 FPS | overhead=-4.26%
 0.5s | 1181.01 FPS | overhead=-1.66%
 1.0s | 1191.01 FPS | overhead=-0.83%


In [24]:
import csv
import json

POWER_INTERVAL = 1.0
POWER_WINDOWS = 8
POWER_SECONDS = 20.0

IDLE_WINDOWS = 3
IDLE_SECONDS = 15.0


def save_rows(path, rows):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=rows[0].keys()
        )
        writer.writeheader()
        writer.writerows(rows)


def measure_idle(label):

    rows = []

    print(f"\n===== IDLE: {label} =====")

    for w in range(IDLE_WINDOWS):

        recorder = DataRecorder(
            power_sensor
        )

        with recorder.record(
            POWER_INTERVAL
        ):
            time.sleep(
                IDLE_SECONDS
            )

        frame = recorder.frame.copy()

        frame.to_csv(
            OUT
            / f"power_idle_{label}_raw_{w+1:02d}.csv"
        )

        values = np.asarray(
            frame[power_sensor.name],
            dtype=np.float64
        )

        values = values[
            np.isfinite(values)
        ]

        row = {
            "window": w + 1,
            "samples": len(values),
            "power_mean_w":
                float(np.mean(values)),
            "power_std_w":
                float(np.std(values, ddof=1))
        }

        rows.append(row)

        print(
            f"{w+1}/{IDLE_WINDOWS} | "
            f"{row['power_mean_w']:.4f} W | "
            f"samples={row['samples']}"
        )

    save_rows(
        OUT / f"power_idle_{label}.csv",
        rows
    )

    idle_means = np.array([
        r["power_mean_w"]
        for r in rows
    ])

    return {
        "windows": rows,
        "statistics":
            cycle_statistics(
                idle_means,
                SEED + 6000
            ),
        "mean_w":
            float(np.mean(idle_means))
    }


def measure_power_scenario(
    scenario,
    idle_power
):

    print(
        f"\n===== POWER: {scenario} ====="
    )

    rows = []

    if scenario == "saturated":
        input_buffer[:] = packed_subset[0]

    # warm-up
    for i in range(WARMUP):

        pos = i % N_IMAGES

        if scenario == "inference_only":

            input_buffer[:] = (
                packed_subset[pos]
            )

            dma_inference_only()

            _ = int(
                np.argmax(
                    decode_logits()
                )
            )

        elif scenario == "end_to_end":

            pack_image_u8(
                x_test[
                    subset_idx[pos]
                ],
                input_buffer
            )

            dma_inference_only()

            _ = int(
                np.argmax(
                    decode_logits()
                )
            )

        elif scenario == "saturated":

            dma_inference_only()


    for w in range(POWER_WINDOWS):

        recorder = DataRecorder(
            power_sensor
        )

        count = 0
        pos = 0

        with recorder.record(
            POWER_INTERVAL
        ):

            t0 = time.perf_counter()

            while (
                time.perf_counter()
                - t0
                < POWER_SECONDS
            ):

                k = pos % N_IMAGES

                if scenario == "inference_only":

                    input_buffer[:] = (
                        packed_subset[k]
                    )

                    dma_inference_only()

                    _ = int(
                        np.argmax(
                            decode_logits()
                        )
                    )

                elif scenario == "end_to_end":

                    pack_image_u8(
                        x_test[
                            subset_idx[k]
                        ],
                        input_buffer
                    )

                    dma_inference_only()

                    _ = int(
                        np.argmax(
                            decode_logits()
                        )
                    )

                elif scenario == "saturated":

                    dma_inference_only()

                count += 1
                pos += 1

            elapsed = (
                time.perf_counter()
                - t0
            )

        frame = recorder.frame.copy()

        frame.to_csv(
            OUT
            / f"power_{scenario}_raw_{w+1:02d}.csv"
        )

        values = np.asarray(
            frame[power_sensor.name],
            dtype=np.float64
        )

        values = values[
            np.isfinite(values)
        ]

        active_power = float(
            np.mean(values)
        )

        fps = count / elapsed

        dynamic_power = max(
            active_power
            - idle_power,
            0.0
        )

        energy_total = (
            active_power
            / fps
            * 1000.0
        )

        energy_dynamic = (
            dynamic_power
            / fps
            * 1000.0
        )

        row = {
            "window": w + 1,
            "count": count,
            "elapsed_s": elapsed,
            "samples": len(values),

            "fps": fps,

            "idle_power_w":
                idle_power,

            "active_power_w":
                active_power,

            "dynamic_power_w":
                dynamic_power,

            "energy_total_mj":
                energy_total,

            "energy_dynamic_mj":
                energy_dynamic
        }

        rows.append(row)

        print(
            f"{w+1}/{POWER_WINDOWS} | "
            f"FPS={fps:.2f} | "
            f"P={active_power:.4f} W | "
            f"Pdyn={dynamic_power:.4f} W | "
            f"E={energy_total:.4f} mJ | "
            f"Edyn={energy_dynamic:.4f} mJ"
        )

    save_rows(
        OUT / f"power_{scenario}.csv",
        rows
    )

    result = {}

    for key in [
        "fps",
        "active_power_w",
        "dynamic_power_w",
        "energy_total_mj",
        "energy_dynamic_mj"
    ]:

        result[key] = (
            cycle_statistics(
                [
                    r[key]
                    for r in rows
                ],
                SEED + 7000
            )
        )

    return {
        "windows": rows,
        "statistics": result
    }

In [26]:
import csv
import json
import time
import numpy as np
from pynq.pmbus import DataRecorder

POWER_INTERVAL = 1.0
POWER_WINDOWS = 8
POWER_SECONDS = 20.0

IDLE_WINDOWS = 3
IDLE_SECONDS = 15.0


def save_rows(path, rows):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=rows[0].keys()
        )
        writer.writeheader()
        writer.writerows(rows)


def measure_idle(scenario):

    rows = []

    print(f"\n===== IDLE: {scenario} =====")

    for w in range(IDLE_WINDOWS):

        rec = DataRecorder(power_sensor)

        with rec.record(POWER_INTERVAL):
            time.sleep(IDLE_SECONDS)

        frame = rec.frame.copy()

        frame.to_csv(
            OUT / f"power_idle_{scenario}_raw_{w+1:02d}.csv"
        )

        values = np.asarray(
            frame[power_sensor.name],
            dtype=np.float64
        )

        values = values[np.isfinite(values)]

        row = {
            "window": w + 1,
            "samples": len(values),
            "power_mean_w": float(np.mean(values)),
            "power_std_w": float(np.std(values, ddof=1))
        }

        rows.append(row)

        print(
            f"{w+1}/{IDLE_WINDOWS} | "
            f"{row['power_mean_w']:.4f} W | "
            f"samples={row['samples']}"
        )

    save_rows(
        OUT / f"power_idle_{scenario}.csv",
        rows
    )

    means = np.asarray([
        r["power_mean_w"]
        for r in rows
    ])

    return {
        "windows": rows,
        "mean_w": float(np.mean(means)),
        "statistics": cycle_statistics(
            means,
            SEED + 6000
        )
    }


def measure_power_scenario(
    scenario,
    idle_power
):

    rows = []

    print(f"\n===== POWER: {scenario} =====")

    # --------------------------------------------------
    # Warm-up
    # --------------------------------------------------

    if scenario == "saturated":

        input_buffer[:] = packed_subset[0]

        for _ in range(WARMUP):
            dma_inference_only()

    else:

        for i in range(WARMUP):

            pos = i % N_IMAGES

            if scenario == "inference_only":

                input_buffer[:] = packed_subset[pos]

                dma_inference_only()

                _ = int(
                    np.argmax(
                        decode_logits()
                    )
                )

            elif scenario == "end_to_end":

                pack_image_u8(
                    x_test[subset_idx[pos]],
                    input_buffer
                )

                dma_inference_only()

                _ = int(
                    np.argmax(
                        decode_logits()
                    )
                )

    # --------------------------------------------------
    # Janelas de medição
    # --------------------------------------------------

    for w in range(POWER_WINDOWS):

        if scenario == "saturated":
            input_buffer[:] = packed_subset[0]

        rec = DataRecorder(power_sensor)

        count = 0
        pos = 0

        with rec.record(POWER_INTERVAL):

            t0 = time.perf_counter()

            while (
                time.perf_counter() - t0
                < POWER_SECONDS
            ):

                k = pos % N_IMAGES

                if scenario == "inference_only":

                    input_buffer[:] = packed_subset[k]

                    dma_inference_only()

                    _ = int(
                        np.argmax(
                            decode_logits()
                        )
                    )

                elif scenario == "end_to_end":

                    pack_image_u8(
                        x_test[subset_idx[k]],
                        input_buffer
                    )

                    dma_inference_only()

                    _ = int(
                        np.argmax(
                            decode_logits()
                        )
                    )

                elif scenario == "saturated":

                    dma_inference_only()

                count += 1
                pos += 1

            elapsed = (
                time.perf_counter() - t0
            )

        frame = rec.frame.copy()

        frame.to_csv(
            OUT
            / f"power_{scenario}_raw_{w+1:02d}.csv"
        )

        values = np.asarray(
            frame[power_sensor.name],
            dtype=np.float64
        )

        values = values[np.isfinite(values)]

        active_power = float(
            np.mean(values)
        )

        fps = count / elapsed

        dynamic_power = max(
            active_power - idle_power,
            0.0
        )

        total_energy = (
            active_power
            / fps
            * 1000.0
        )

        dynamic_energy = (
            dynamic_power
            / fps
            * 1000.0
        )

        row = {
            "window": w + 1,
            "count": count,
            "elapsed_s": elapsed,
            "samples": len(values),

            "fps": fps,

            "idle_power_w":
                idle_power,

            "active_power_w":
                active_power,

            "dynamic_power_w":
                dynamic_power,

            "energy_total_mj":
                total_energy,

            "energy_dynamic_mj":
                dynamic_energy
        }

        rows.append(row)

        print(
            f"{w+1}/{POWER_WINDOWS} | "
            f"FPS={fps:.2f} | "
            f"P={active_power:.4f} W | "
            f"Pdyn={dynamic_power:.4f} W | "
            f"E={total_energy:.4f} mJ | "
            f"Edyn={dynamic_energy:.4f} mJ"
        )

    save_rows(
        OUT / f"power_{scenario}.csv",
        rows
    )

    statistics = {}

    for key in [
        "fps",
        "active_power_w",
        "dynamic_power_w",
        "energy_total_mj",
        "energy_dynamic_mj"
    ]:

        statistics[key] = cycle_statistics(
            [r[key] for r in rows],
            SEED + 7000
        )

    return {
        "windows": rows,
        "statistics": statistics
    }


power_results = {}

for scenario in [
    "inference_only",
    "end_to_end",
    "saturated"
]:

    idle = measure_idle(scenario)

    active = measure_power_scenario(
        scenario,
        idle["mean_w"]
    )

    power_results[scenario] = {
        "idle": idle,
        "active": active
    }


# ------------------------------------------------------
# Quantificar perturbação da telemetria
# ------------------------------------------------------

clean_fps = {
    "inference_only":
        inf_effective_stats["mean"],

    "end_to_end":
        e2e_fps_stats["mean"],

    "saturated":
        sat_stats["mean"]
}

telemetry_overhead = {}

for scenario in clean_fps:

    measured = (
        power_results[scenario]
        ["active"]
        ["statistics"]
        ["fps"]
        ["mean"]
    )

    telemetry_overhead[scenario] = (
        (measured / clean_fps[scenario] - 1.0)
        * 100.0
    )

power_results[
    "telemetry_overhead_percent"
] = telemetry_overhead


with open(
    OUT / "POWER_SUMMARY.json",
    "w"
) as f:

    json.dump(
        power_results,
        f,
        indent=2
    )


print("\n===== OVERHEAD DA TELEMETRIA =====")

for k, v in telemetry_overhead.items():

    print(
        f"{k:16s}: {v:+.3f}%"
    )


print("\nPOWER_SUMMARY salvo em:")
print(OUT / "POWER_SUMMARY.json")


===== IDLE: inference_only =====
1/3 | 10.6064 W | samples=15
2/3 | 10.6139 W | samples=15
3/3 | 10.6080 W | samples=15

===== POWER: inference_only =====
1/8 | FPS=904.91 | P=12.2248 W | Pdyn=1.6153 W | E=13.5093 mJ | Edyn=1.7851 mJ
2/8 | FPS=906.72 | P=12.2066 W | Pdyn=1.5972 W | E=13.4623 mJ | Edyn=1.7615 mJ
3/8 | FPS=903.79 | P=12.2322 W | Pdyn=1.6228 W | E=13.5344 mJ | Edyn=1.7955 mJ
4/8 | FPS=906.41 | P=12.2422 W | Pdyn=1.6328 W | E=13.5063 mJ | Edyn=1.8014 mJ
5/8 | FPS=906.61 | P=12.2348 W | Pdyn=1.6253 W | E=13.4951 mJ | Edyn=1.7928 mJ
6/8 | FPS=906.34 | P=12.2223 W | Pdyn=1.6129 W | E=13.4853 mJ | Edyn=1.7796 mJ
7/8 | FPS=905.10 | P=12.2292 W | Pdyn=1.6198 W | E=13.5114 mJ | Edyn=1.7896 mJ
8/8 | FPS=906.86 | P=12.2343 W | Pdyn=1.6249 W | E=13.4908 mJ | Edyn=1.7918 mJ

===== IDLE: end_to_end =====
1/3 | 10.6340 W | samples=15
2/3 | 10.6240 W | samples=15
3/3 | 10.6231 W | samples=15

===== POWER: end_to_end =====
1/8 | FPS=618.51 | P=11.8167 W | Pdyn=1.1897 W | E=19.1053 mJ | 

In [29]:
from pathlib import Path
import json
import numpy as np

# ============================================================
# Carregar resultados já salvos
# ============================================================

summary = json.loads(
    (OUT / "SUMMARY.json").read_text()
)

power = json.loads(
    (OUT / "POWER_SUMMARY.json").read_text()
)

IL = summary["inference_only"]["latency"]
IF = summary["inference_only"]["effective_fps"]
IP = summary["inference_only"]["path_fps"]

EL = summary["end_to_end"]["latency"]
EF = summary["end_to_end"]["effective_fps"]

SF = summary["saturated"]["fps"]

PI = power["inference_only"]
PE = power["end_to_end"]
PS = power["saturated"]


def ps(P, key):
    return P["active"]["statistics"][key]


def mean(P, key):
    return float(ps(P, key)["mean"])


def ci(P, key):
    s = ps(P, key)
    return (
        float(s["ci95_low"]),
        float(s["ci95_high"])
    )


def fmt(x, n=4):
    return f"{float(x):.{n}f}"


def perf_ci(s, key_low="ci95_low", key_high="ci95_high", n=2):
    return (
        f"[{float(s[key_low]):.{n}f}; "
        f"{float(s[key_high]):.{n}f}]"
    )


L = []

# ============================================================
# CABEÇALHO
# ============================================================

L += [
"# ResNet8 hls4ml — Benchmark na AMD/Xilinx ZCU104",
"",
"## 1. Objetivo",
"",
"Este diretório contém os resultados finais de desempenho e eficiência "
"energética da implementação da ResNet8 para o conjunto CIFAR-10 utilizando "
"hls4ml na AMD/Xilinx ZCU104.",
"",
"O benchmark foi dividido em três cenários com fronteiras de medição distintas:",
"",
"1. **Inference batch 1 síncrono** — principal cenário para comparação com CPU/GPU;",
"2. **End-to-end host-to-class** — inclui pré-processamento e pós-processamento;",
"3. **Throughput sustentado do caminho acelerado** — caracteriza a capacidade "
"máxima sustentada DMA + FPGA + DMA.",
"",
"Os três valores não representam a mesma fronteira experimental e, portanto, "
"não devem ser utilizados de forma intercambiável.",
"",
]

# ============================================================
# CONFIGURAÇÃO
# ============================================================

L += [
"## 2. Plataforma e configuração",
"",
"- Plataforma: AMD/Xilinx ZCU104",
"- FPGA/SoC: XCZU7EV-FFVC1156-2-E",
"- Modelo: ResNet8",
"- Dataset: CIFAR-10",
"- Entrada: 32 × 32 × 3",
"- Classes: 10",
"- Parâmetros: 78.714",
"- Saída: 10 logits",
"- Softmax: removida",
"- Classificação: `argmax(logits)`",
"- Batch: 1",
"- Clock do acelerador: 100 MHz",
"- Backend hls4ml: Vitis",
"- IOType: `io_stream`",
"- Strategy: `Resource`",
"- ConvImplementation: `LineBuffer`",
"- FIFO optimization: habilitada",
"- Reuse Factor máximo: 288",
"- Precisão: `ap_fixed<22,12,AP_RND_CONV,AP_SAT>`",
"",
]

# ============================================================
# ACURÁCIA
# ============================================================

L += [
"## 3. Validação de acurácia",
"",
"A acurácia foi validada separadamente utilizando as 10.000 imagens oficiais "
"do conjunto de teste do CIFAR-10.",
"",
"- Imagens únicas: **10.000**",
"- Acertos: **7.492**",
"- Acurácia top-1: **74,9200%**",
"- IC95% de Wilson: **[74,0609%; 75,7599%]**",
"",
"As repetições utilizadas para benchmarking não foram consideradas novas "
"observações de acurácia.",
"",
]

# ============================================================
# CENÁRIO 1
# ============================================================

L += [
"# 4. Cenário 1 — Inference batch 1 síncrono",
"",
"## 4.1 Fronteira de medição",
"",
"A imagem encontra-se previamente normalizada, quantizada e empacotada.",
"",
"O throughput efetivo corresponde ao seguinte fluxo:",
"",
"```text",
"entrada pré-processada",
"    ↓",
"cópia para PynqBuffer",
"    ↓",
"DMA MM2S",
"    ↓",
"ResNet8 hls4ml",
"    ↓",
"DMA S2MM",
"    ↓",
"decode dos logits",
"    ↓",
"argmax",
"    ↓",
"próxima imagem",
"```",
"",
"A latência individual `inference-only` possui uma fronteira menor. "
"O cronômetro começa após a entrada já estar presente no PynqBuffer e cobre:",
"",
"```text",
"DMA MM2S → ResNet8 FPGA → DMA S2MM → wait()",
"```",
"",
"Foram utilizadas 5.000 imagens estratificadas e 20 ciclos, totalizando "
"**100.000 inferências temporizadas**.",
"",
"## 4.2 Resultados — Inference batch 1",
"",
"| Métrica | Resultado |",
"|---|---:|",
f"| Inferências temporizadas | {IL['samples']} |",
f"| Latência média | **{fmt(IL['mean_ms'])} ms** |",
f"| Mediana | {fmt(IL['median_ms'])} ms |",
f"| Desvio-padrão | {fmt(IL['std_ms'])} ms |",
f"| CV das latências | {fmt(IL['cv_percent'],3)}% |",
f"| p95 | **{fmt(IL['p95_ms'])} ms** |",
f"| p99 | **{fmt(IL['p99_ms'])} ms** |",
f"| Mínimo | {fmt(IL['min_ms'])} ms |",
f"| Máximo | {fmt(IL['max_ms'])} ms |",
f"| Throughput efetivo | **{fmt(IF['mean'],2)} FPS** |",
f"| Desvio do throughput | {fmt(IF['sd'],2)} FPS |",
f"| CV do throughput | {fmt(IF['cv_percent'],3)}% |",
f"| IC95% throughput | {perf_ci(IF)} FPS |",
f"| IC95% bootstrap | {perf_ci(IF,'bootstrap_ci95_low','bootstrap_ci95_high')} FPS |",
f"| Drift temporal | {fmt(IF['drift_percent'],3)}% |",
f"| Taxa equivalente pela latência | {fmt(IP['mean'],2)} FPS |",
f"| Potência idle | **{fmt(PI['idle']['mean_w'])} W** |",
f"| Potência ativa | **{fmt(mean(PI,'active_power_w'))} W** |",
f"| Potência dinâmica | **{fmt(mean(PI,'dynamic_power_w'))} W** |",
f"| Energia total/inferência | **{fmt(mean(PI,'energy_total_mj'))} mJ** |",
f"| Energia dinâmica/inferência | **{fmt(mean(PI,'energy_dynamic_mj'))} mJ** |",
f"| FPS durante telemetria | {fmt(mean(PI,'fps'),2)} FPS |",
f"| Overhead da telemetria | {fmt(power['telemetry_overhead_percent']['inference_only'],3)}% |",
"",
]

# ============================================================
# CENÁRIO 2
# ============================================================

L += [
"# 5. Cenário 2 — End-to-end",
"",
"## 5.1 Fronteira de medição",
"",
"O cenário end-to-end parte de uma imagem CIFAR-10 `uint8` já presente "
"na memória RAM e termina na classe prevista.",
"",
"```text",
"imagem uint8 em RAM",
"    ↓",
"conversão float32",
"    ↓",
"normalização /255",
"    ↓",
"quantização ap_fixed<22,12>",
"    ↓",
"packing",
"    ↓",
"PynqBuffer",
"    ↓",
"DMA MM2S",
"    ↓",
"ResNet8 hls4ml",
"    ↓",
"DMA S2MM",
"    ↓",
"decode dos logits",
"    ↓",
"argmax",
"    ↓",
"classe prevista",
"```",
"",
"Não entram no tempo: carregamento do dataset a partir do armazenamento, "
"programação do FPGA, criação do Overlay, alocação inicial dos buffers e warm-up.",
"",
"## 5.2 Resultados — End-to-end",
"",
"| Métrica | Resultado |",
"|---|---:|",
f"| Inferências temporizadas | {EL['samples']} |",
f"| Latência média | **{fmt(EL['mean_ms'])} ms** |",
f"| Mediana | {fmt(EL['median_ms'])} ms |",
f"| Desvio-padrão | {fmt(EL['std_ms'])} ms |",
f"| CV das latências | {fmt(EL['cv_percent'],3)}% |",
f"| p95 | **{fmt(EL['p95_ms'])} ms** |",
f"| p99 | **{fmt(EL['p99_ms'])} ms** |",
f"| Mínimo | {fmt(EL['min_ms'])} ms |",
f"| Máximo | {fmt(EL['max_ms'])} ms |",
f"| Throughput efetivo | **{fmt(EF['mean'],2)} FPS** |",
f"| Desvio do throughput | {fmt(EF['sd'],2)} FPS |",
f"| CV do throughput | {fmt(EF['cv_percent'],3)}% |",
f"| IC95% throughput | {perf_ci(EF)} FPS |",
f"| IC95% bootstrap | {perf_ci(EF,'bootstrap_ci95_low','bootstrap_ci95_high')} FPS |",
f"| Drift temporal | {fmt(EF['drift_percent'],3)}% |",
f"| Potência idle | **{fmt(PE['idle']['mean_w'])} W** |",
f"| Potência ativa | **{fmt(mean(PE,'active_power_w'))} W** |",
f"| Potência dinâmica | **{fmt(mean(PE,'dynamic_power_w'))} W** |",
f"| Energia total/inferência | **{fmt(mean(PE,'energy_total_mj'))} mJ** |",
f"| Energia dinâmica/inferência | **{fmt(mean(PE,'energy_dynamic_mj'))} mJ** |",
f"| FPS durante telemetria | {fmt(mean(PE,'fps'),2)} FPS |",
f"| Overhead da telemetria | {fmt(power['telemetry_overhead_percent']['end_to_end'],3)}% |",
"",
]

# ============================================================
# CENÁRIO 3
# ============================================================

L += [
"# 6. Cenário 3 — Acelerador saturado",
"",
"## 6.1 Fronteira de medição",
"",
"Este cenário caracteriza a capacidade máxima sustentada do caminho acelerado. "
"A entrada já está completamente empacotada e presente no PynqBuffer.",
"",
"Durante a janela de benchmark é repetido somente:",
"",
"```text",
"DMA MM2S → ResNet8 hls4ml → DMA S2MM → wait() → próxima execução",
"```",
"",
"Não entram: normalização, quantização, packing, cópia de uma nova imagem, "
"decode dos logits ou `argmax`.",
"",
"Esse ensaio continua sendo serial e síncrono. Não foram utilizadas várias "
"inferências simultâneas, double buffering ou várias requisições pendentes.",
"",
"Por esse motivo, o resultado deve ser apresentado como uma métrica adicional "
"da FPGA e não substituir o throughput batch 1 na comparação CPU/GPU.",
"",
"## 6.2 Resultados — Acelerador saturado",
"",
"| Métrica | Resultado |",
"|---|---:|",
f"| Janelas de desempenho | 8 |",
f"| Duração por janela | 10 s |",
f"| Throughput sustentado | **{fmt(SF['mean'],2)} FPS** |",
f"| Desvio-padrão | {fmt(SF['sd'],2)} FPS |",
f"| CV | {fmt(SF['cv_percent'],3)}% |",
f"| IC95% | {perf_ci(SF)} FPS |",
f"| IC95% bootstrap | {perf_ci(SF,'bootstrap_ci95_low','bootstrap_ci95_high')} FPS |",
f"| Drift temporal | {fmt(SF['drift_percent'],3)}% |",
f"| Potência idle | **{fmt(PS['idle']['mean_w'])} W** |",
f"| Potência ativa | **{fmt(mean(PS,'active_power_w'))} W** |",
f"| Potência dinâmica | **{fmt(mean(PS,'dynamic_power_w'))} W** |",
f"| Energia total/inferência | **{fmt(mean(PS,'energy_total_mj'))} mJ** |",
f"| Energia dinâmica/inferência | **{fmt(mean(PS,'energy_dynamic_mj'))} mJ** |",
f"| FPS durante telemetria | {fmt(mean(PS,'fps'),2)} FPS |",
f"| Overhead da telemetria | {fmt(power['telemetry_overhead_percent']['saturated'],3)}% |",
"",
]

# ============================================================
# TABELA RESUMO DOS 3 CENÁRIOS
# ============================================================

L += [
"# 7. Síntese dos três cenários",
"",
"| Métrica | Inference batch 1 | End-to-end | Acelerador saturado |",
"|---|---:|---:|---:|",
f"| Latência média | **{fmt(IL['mean_ms'])} ms** | **{fmt(EL['mean_ms'])} ms** | — |",
f"| Mediana | {fmt(IL['median_ms'])} ms | {fmt(EL['median_ms'])} ms | — |",
f"| p95 | {fmt(IL['p95_ms'])} ms | {fmt(EL['p95_ms'])} ms | — |",
f"| p99 | {fmt(IL['p99_ms'])} ms | {fmt(EL['p99_ms'])} ms | — |",
f"| Throughput | **{fmt(IF['mean'],2)} FPS** | **{fmt(EF['mean'],2)} FPS** | **{fmt(SF['mean'],2)} FPS** |",
f"| Potência idle | {fmt(PI['idle']['mean_w'])} W | {fmt(PE['idle']['mean_w'])} W | {fmt(PS['idle']['mean_w'])} W |",
f"| Potência ativa | {fmt(mean(PI,'active_power_w'))} W | {fmt(mean(PE,'active_power_w'))} W | {fmt(mean(PS,'active_power_w'))} W |",
f"| Potência dinâmica | {fmt(mean(PI,'dynamic_power_w'))} W | {fmt(mean(PE,'dynamic_power_w'))} W | {fmt(mean(PS,'dynamic_power_w'))} W |",
f"| Energia total/inf. | {fmt(mean(PI,'energy_total_mj'))} mJ | {fmt(mean(PE,'energy_total_mj'))} mJ | {fmt(mean(PS,'energy_total_mj'))} mJ |",
f"| Energia dinâmica/inf. | {fmt(mean(PI,'energy_dynamic_mj'))} mJ | {fmt(mean(PE,'energy_dynamic_mj'))} mJ | {fmt(mean(PS,'energy_dynamic_mj'))} mJ |",
"",
]

# ============================================================
# TELEMETRIA
# ============================================================

L += [
"# 8. Metodologia energética",
"",
"A potência foi medida utilizando o sensor PMBus `12V_power` da ZCU104.",
"",
"A potência dinâmica foi definida como:",
"",
"```text",
"P_dinâmica = P_ativa - P_idle",
"```",
"",
"A energia por inferência foi obtida a partir da potência e do throughput "
"observados durante a mesma janela de aquisição:",
"",
"```text",
"E_total = P_ativa / FPS",
"E_dinâmica = P_dinâmica / FPS",
"```",
"",
"A frequência final de aquisição utilizada foi de 1 s.",
"",
"Antes da coleta definitiva foi realizado um ensaio para avaliar o impacto "
"da telemetria sobre o throughput saturado:",
"",
"| Intervalo PMBus | FPS | Overhead |",
"|---:|---:|---:|",
"| 0,1 s | 1104,98 | -7,99% |",
"| 0,2 s | 1149,77 | -4,26% |",
"| 0,5 s | 1181,01 | -1,66% |",
"| 1,0 s | 1191,01 | -0,83% |",
"",
"A amostragem de 1 s foi selecionada por apresentar baixa perturbação e ainda "
"permitir múltiplas observações durante cada janela de potência.",
"",
"Na coleta energética definitiva, os impactos medidos foram:",
"",
f"- Inference batch 1: **{fmt(power['telemetry_overhead_percent']['inference_only'],3)}%**",
f"- End-to-end: **{fmt(power['telemetry_overhead_percent']['end_to_end'],3)}%**",
f"- Acelerador saturado: **{fmt(power['telemetry_overhead_percent']['saturated'],3)}%**",
"",
]

# ============================================================
# ESTATÍSTICA
# ============================================================

L += [
"# 9. Validação estatística",
"",
"Nenhum outlier foi removido.",
"",
"Foram preservadas todas as observações de latência. Para avaliar estabilidade "
"temporal, os ciclos ou janelas foram utilizados como unidade estatística.",
"",
"Foram reportados:",
"",
"- média;",
"- mediana;",
"- desvio-padrão amostral;",
"- coeficiente de variação (CV);",
"- p95;",
"- p99;",
"- mínimo;",
"- máximo;",
"- intervalo de confiança de 95%;",
"- intervalo de confiança bootstrap;",
"- drift entre o início e o final do ensaio.",
"",
"Os coeficientes de variação extremamente baixos entre os ciclos demonstram "
"boa repetibilidade das medições.",
"",
]

# ============================================================
# COMPARAÇÃO CPU/GPU
# ============================================================

L += [
"# 10. Uso dos resultados na comparação CPU × GPU × FPGA",
"",
"Para a comparação principal entre plataformas recomenda-se:",
"",
"### Latência",
"",
"Utilizar a latência `inference-only`, pois a entrada já está no formato "
"esperado pelo runtime/acelerador antes do início do cronômetro.",
"",
f"Resultado ZCU104: **{fmt(IL['mean_ms'])} ms**.",
"",
"### Throughput batch 1",
"",
"Utilizar o throughput efetivo do cenário `inference batch 1`, pois ele inclui "
"a orquestração host necessária para processar sequencialmente imagens diferentes.",
"",
f"Resultado ZCU104: **{fmt(IF['mean'],2)} FPS**.",
"",
"### End-to-end",
"",
"Utilizar somente em comparação com CPU/GPU quando essas plataformas também "
"forem medidas a partir da imagem `uint8` em RAM, incluindo normalização e "
"pré-processamento.",
"",
f"Resultado ZCU104: **{fmt(EL['mean_ms'])} ms / {fmt(EF['mean'],2)} FPS**.",
"",
"### Capacidade sustentada do caminho acelerado",
"",
f"O resultado de **{fmt(SF['mean'],2)} FPS** caracteriza a capacidade sustentada "
"DMA + FPGA + DMA com a entrada já pronta. Essa métrica deve permanecer separada "
"da comparação principal de throughput batch 1.",
"",
]

# ============================================================
# REPRODUTIBILIDADE
# ============================================================

L += [
"# 11. Reprodutibilidade",
"",
"O diretório do experimento preserva os arquivos brutos e agregados necessários "
"para análise posterior:",
"",
"- `SUMMARY.json`;",
"- `POWER_SUMMARY.json`;",
"- `cycles_inference_only.csv`;",
"- `cycles_end_to_end.csv`;",
"- `latencias_inference_only.npy`;",
"- `latencias_end_to_end.npy`;",
"- `fps_inference_only_effective.npy`;",
"- `fps_inference_only_path.npy`;",
"- `fps_end_to_end.npy`;",
"- `fps_saturated.npy`;",
"- `subset_indices.npy`;",
"- `reference_predictions.npy`;",
"- arquivos `power_*.csv`;",
"- amostras brutas `power_*_raw_*.csv`;",
"- bitstream `.bit`;",
"- hardware handoff `.hwh`;",
"- dataset utilizado;",
"- notebook do benchmark;",
"- hashes SHA-256 dos artefatos.",
"",
]

README_TEXT = "\n".join(L) + "\n"

README_PATH = OUT / "README.md"
README_PATH.write_text(
    README_TEXT,
    encoding="utf-8"
)

print("README criado com sucesso:")
print(README_PATH)
print()
print("Linhas:", len(L))
print("Bytes :", README_PATH.stat().st_size)

README criado com sucesso:
/home/xilinx/jupyter_notebooks/resnet8_hls_ip11/resultados_intermediario_20250504_103336/README.md

Linhas: 331
Bytes : 10108


In [30]:
print(
    (OUT / "README.md").read_text(
        encoding="utf-8"
    )
)

# ResNet8 hls4ml — Benchmark na AMD/Xilinx ZCU104

## 1. Objetivo

Este diretório contém os resultados finais de desempenho e eficiência energética da implementação da ResNet8 para o conjunto CIFAR-10 utilizando hls4ml na AMD/Xilinx ZCU104.

O benchmark foi dividido em três cenários com fronteiras de medição distintas:

1. **Inference batch 1 síncrono** — principal cenário para comparação com CPU/GPU;
2. **End-to-end host-to-class** — inclui pré-processamento e pós-processamento;
3. **Throughput sustentado do caminho acelerado** — caracteriza a capacidade máxima sustentada DMA + FPGA + DMA.

Os três valores não representam a mesma fronteira experimental e, portanto, não devem ser utilizados de forma intercambiável.

## 2. Plataforma e configuração

- Plataforma: AMD/Xilinx ZCU104
- FPGA/SoC: XCZU7EV-FFVC1156-2-E
- Modelo: ResNet8
- Dataset: CIFAR-10
- Entrada: 32 × 32 × 3
- Classes: 10
- Parâmetros: 78.714
- Saída: 10 logits
- Softmax: removida
- Classificação: `argmax(logits)`
- Batch